# Sentence-aware chunking with overlap (Step 1)

Replaces the fixed 250-word slicing with a chunker that:

- **respects sentence boundaries** — a chunk never starts or ends mid-sentence;
- **targets a token budget** (~256 tokens) measured with the *real* embedding
  tokenizer, not a word count;
- **adds ~40-token overlap** between consecutive chunks, carried at the
  sentence level, so an answer that straddles a boundary still lands whole in
  at least one chunk.

Design notes:
- **Tokenizer:** the XLM-RoBERTa tokenizer, which `intfloat/multilingual-e5-base`
  *and* `BAAI/bge-m3` both use — so chunk sizes stay consistent across the two
  models you're comparing.
- **Sentence splitter:** spaCy's fast rule-based `sentencizer`. Because the
  chunker repacks sentences into token windows anyway, occasional over-splitting
  at German abbreviations (`z.B.`) is harmless, and it avoids running the
  statistical parser over the 472-page document.

Output: `vector_store/chunks_v2.jsonl` (kept separate from the existing
`chunks.jsonl` so you can compare). Topic tags are carried over via the current
keyword method for now — **Step 2 replaces them with real LDA inference.**

Requirements: `pip install spacy transformers` (both already present via
`sentence-transformers`); no extra model download needed for the sentencizer.


## Config

In [ ]:
import re
import json
import unicodedata
from pathlib import Path
from collections import Counter

INPUT_TEXT_DIR = Path("Input text files")            # the 12 extracted STEK docs
SCRAPED_MANIFEST = Path("scraped_data_topics/manifest.json")
LDA_TOPICS_PATH = Path("STEK2035-chatbot/pdfs/stek_output/texts/stek_results/lda_topics_k5.txt")

OUT_DIR = Path("vector_store")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_V2_PATH = OUT_DIR / "chunks_v2.jsonl"

# tokenizer shared by e5 and bge-m3, so chunk sizes match both models
TOKENIZER_NAME = "intfloat/multilingual-e5-base"

TARGET_TOKENS = 256      # target chunk size
OVERLAP_TOKENS = 40      # ~15% overlap, carried at sentence granularity
MIN_TOKENS = 50          # drop trailing chunks smaller than this (merged back instead)


## Load the tokenizer and the sentence splitter

In [ ]:
import spacy
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

# Fast, rule-based German sentence segmentation. No statistical model needed.
nlp = spacy.blank("de")
nlp.add_pipe("sentencizer")
nlp.max_length = 3_000_000   # the largest doc is well under this

def split_sentences(text: str) -> list[str]:
    doc = nlp(text)
    return [s.text.strip() for s in doc.sents if s.text.strip()]

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

print("Tokenizer:", TOKENIZER_NAME)
print("Sentence splitter ready:", nlp.pipe_names)


## Shared cleaning + topic-tagging helpers

Same boilerplate cleaning and LDA-keyword tagging as the earlier pipeline, so
the only thing that changes in this step is *how the text is cut into chunks*.


In [ ]:
def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = re.sub(r"([a-zäöüß])-\s*\n\s*([a-zäöüß])", r"\1\2", text)  # de-hyphenate line breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def strip_header(raw: str) -> str:
    """Scraped files have a metadata header, then a blank line, then the body."""
    norm = raw.replace("\r\n", "\n").replace("\r", "\n")
    parts = norm.split("\n\n", 1)
    return parts[1] if len(parts) == 2 else norm


def parse_lda_topics(path: Path) -> dict[int, list[str]]:
    topics, pat = {}, re.compile(r"Topic\s+(\d+):\s*(.+)")
    for line in path.read_text(encoding="utf-8").splitlines():
        m = pat.match(line.strip())
        if m:
            topics[int(m.group(1))] = [w.strip() for w in m.group(2).split(",") if w.strip()]
    return topics


TOPIC_KEYWORDS = parse_lda_topics(LDA_TOPICS_PATH)


def tag_topics(text: str):
    lower = text.lower()
    matched_topics, matched_keywords, seen = [], [], set()
    for topic_id, keywords in sorted(TOPIC_KEYWORDS.items()):
        hit = False
        for kw in keywords:
            needle = kw.replace("_", " ").lower()
            if re.search(rf"\b{re.escape(needle)}\b", lower):
                hit = True
                if kw not in seen:
                    seen.add(kw); matched_keywords.append(kw)
        if hit:
            matched_topics.append(topic_id)
    return matched_topics, matched_keywords, len(matched_keywords)


def norm_key(name: str) -> str:
    name = re.sub(r"%[0-9a-fA-F]{2}", " ", name).rsplit(".", 1)[0]
    name = re.sub(r"(?i)^\d+_pdf_", "", name)
    return re.sub(r"[^a-z0-9]", "", name.lower())


## The sentence-aware chunker with overlap

The algorithm, in plain terms:
1. Split the text into sentences and pre-count each sentence's tokens.
2. Any single sentence longer than the target (a table row, a run-on list) is
   hard-split into token-sized pieces so no chunk can exceed the model's limit.
3. Greedily pack sentences into a chunk until the next one would exceed
   `target_tokens`.
4. To start the next chunk, walk back from the end of the current one and repeat
   the trailing sentences whose tokens sum to ≥ `overlap_tokens` — that's the
   overlap. Progress is guaranteed: if the overlap would swallow the whole chunk
   (a single huge sentence), we start fresh with no overlap.


In [ ]:
def _split_oversized(sentence: str, max_tokens: int) -> list[str]:
    """Hard-split a single over-long 'sentence' into <= max_tokens word-pieces."""
    words = sentence.split()
    pieces, buf = [], []
    for w in words:
        buf.append(w)
        if count_tokens(" ".join(buf)) >= max_tokens:
            pieces.append(" ".join(buf)); buf = []
    if buf:
        pieces.append(" ".join(buf))
    return pieces


def sentence_aware_chunks(text: str, target_tokens: int, overlap_tokens: int,
                          min_tokens: int) -> list[dict]:
    # 1-2: sentences + token counts, splitting any oversized sentence
    raw_sents = split_sentences(text)
    sents, counts = [], []
    for s in raw_sents:
        c = count_tokens(s)
        if c > target_tokens:
            for piece in _split_oversized(s, target_tokens):
                sents.append(piece); counts.append(count_tokens(piece))
        else:
            sents.append(s); counts.append(c)

    if not sents:
        return []

    # 3-4: greedy packing with sentence-level overlap
    chunks, i, N = [], 0, len(sents)
    while i < N:
        cur_tok, j = 0, i
        while j < N and (cur_tok + counts[j] <= target_tokens or j == i):
            cur_tok += counts[j]; j += 1
        chunks.append({
            "text": " ".join(sents[i:j]),
            "sent_start": i, "sent_end": j - 1, "token_count": cur_tok,
        })
        if j >= N:
            break
        # overlap: walk back to include trailing sentences >= overlap_tokens
        ov, k = 0, j
        while k > i and ov < overlap_tokens:
            k -= 1; ov += counts[k]
        i = k if k > i else j   # guarantee forward progress

    # merge a too-small trailing chunk into the previous one
    if len(chunks) >= 2 and chunks[-1]["token_count"] < min_tokens:
        prev, last = chunks[-2], chunks.pop()
        prev["text"] += " " + last["text"]
        prev["sent_end"] = last["sent_end"]
        prev["token_count"] += last["token_count"]
    return chunks


## Quick check — does it respect boundaries, size, and overlap?

In [ ]:
demo = ("Heidelberg plant bezahlbaren Wohnraum. Die Stadt erschließt neue "
        "Wohngebiete auf Konversionsflächen. Der Klimawandel bringt mehr "
        "Hitzewellen und Hochwasser. Grünflächen und Bäume sollen erhalten "
        "bleiben. Der ÖPNV wird ausgebaut, während Parkplätze reduziert werden. "
        "Kinder und ältere Menschen brauchen sichere Wege. Die Altstadt und der "
        "Neckar prägen das Stadtbild und sollen geschützt werden.")

demo_chunks = sentence_aware_chunks(demo, target_tokens=30, overlap_tokens=8, min_tokens=5)
for n, ch in enumerate(demo_chunks):
    print(f"chunk {n}  (sents {ch['sent_start']}-{ch['sent_end']}, {ch['token_count']} tok)")
    print("   ", ch["text"])
print("\nNote how each chunk starts by repeating the tail sentence of the previous one (overlap),")
print("and no chunk ever cuts a sentence in half.")


## Re-chunk all sources → `chunks_v2.jsonl`

In [ ]:
# --- the 12 STEK documents (clean extracted text, no header) ------------------
records = []
doc_keys = set()
for path in sorted(INPUT_TEXT_DIR.glob("*.txt")):
    document_id = path.stem
    doc_keys.add(norm_key(document_id))
    body = clean_text(path.read_text(encoding="utf-8"))
    for cid, ch in enumerate(sentence_aware_chunks(body, TARGET_TOKENS, OVERLAP_TOKENS, MIN_TOKENS)):
        topics, _, score = tag_topics(ch["text"])
        records.append({
            "chunk_uid": f"document::{document_id}::{cid}",
            "source": "document",
            "origin": path.name,
            "chunk_id": cid,
            "position": f"s{ch['sent_start']}-s{ch['sent_end']}",
            "token_count": ch["token_count"],
            "text": ch["text"],
            "topics": topics,                 # keyword-based; replaced by LDA in Step 2
            "relevance_score": score,
        })

# --- scraped website pages (strip header, dedup PDFs that copy the 12 docs) ----
scraped = json.loads(SCRAPED_MANIFEST.read_text(encoding="utf-8"))
DUPLICATE_ALIASES = {"12_pdf_Step_2015_mit_Lesezeichen_mit_Vorwort_E_Wuerzner_s.pdf"}
skipped = 0
for e in scraped:
    tail = e["url"].split("/")[-1]
    is_pdf = e["type"] == "pdf"
    if is_pdf and (norm_key(tail) in doc_keys or tail in DUPLICATE_ALIASES):
        skipped += 1
        continue
    body = clean_text(strip_header(Path(e["path"]).read_text(encoding="utf-8")))
    source = "website_pdf" if is_pdf else "website_html"
    for cid, ch in enumerate(sentence_aware_chunks(body, TARGET_TOKENS, OVERLAP_TOKENS, MIN_TOKENS)):
        topics, _, score = tag_topics(ch["text"])
        records.append({
            "chunk_uid": f"{source}::{tail}::{cid}",
            "source": source,
            "origin": e["url"],
            "chunk_id": cid,
            "position": f"s{ch['sent_start']}-s{ch['sent_end']}",
            "token_count": ch["token_count"],
            "text": ch["text"],
            "topics": topics,
            "relevance_score": score,
        })

with CHUNKS_V2_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Skipped {skipped} duplicate scraped PDFs")
print(f"Wrote {len(records)} chunks -> {CHUNKS_V2_PATH}")
print("By source:", dict(Counter(r["source"] for r in records)))


## Old vs new chunking — did the strategy change anything?

In [ ]:
import numpy as np

tok = np.array([r["token_count"] for r in records])
print("NEW sentence-aware chunks")
print(f"  count:  {len(tok)}")
print(f"  tokens  min/median/mean/max: {tok.min()} / {int(np.median(tok))} / {tok.mean():.0f} / {tok.max()}")
print(f"  % within 128-320 tokens: {100*np.mean((tok>=128)&(tok<=320)):.0f}%")

old_path = OUT_DIR / "chunks.jsonl"
if old_path.exists():
    old = [json.loads(l) for l in old_path.read_text(encoding="utf-8").splitlines()]
    old_tok = np.array([count_tokens(o["text"]) for o in old])
    print("\nOLD fixed 250-word chunks")
    print(f"  count:  {len(old_tok)}")
    print(f"  tokens  min/median/mean/max: {old_tok.min()} / {int(np.median(old_tok))} / {old_tok.mean():.0f} / {old_tok.max()}")
    print("\nThe new chunks cluster tightly around the token target and never split a")
    print("sentence; the old ones vary more and cut mid-sentence at word 250.")


## Next step

`chunks_v2.jsonl` is the new chunk set. Before embedding it, do **Step 2**:
replace the keyword-based `topics` with real LDA inference (dominant topic +
topic distribution) stored in each chunk's metadata. Then embed `chunks_v2`
with E5 and BGE and run the retrieval comparison.
